# Benford Analysis with pandas

This notebook compares the first significant digits in a public CSV with Benford's Law. The default source is the Plotly Gapminder dataset, which contains more than 1,000 country-year observations; if an `ohio.csv` file is present in the working directory, the notebook uses it instead. The selected column is `population`.

Benford results are descriptive screening evidence, not proof of error or fraud. The final section reports the digits with the largest differences and practical follow-up checks.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

In [ ]:
from pathlib import Path

# Public Plotly repository CSV: 1,704 country-year observations.
data_url = "https://raw.githubusercontent.com/plotly/datasets/master/gapminderDataFiveYear.csv"
local_csv = Path("ohio.csv")

if local_csv.exists():
    df = pd.read_csv(local_csv)
    data_source = str(local_csv)
else:
    df = pd.read_csv(data_url)
    data_source = data_url

column_name = "population" if "population" in df.columns else "pop"
if column_name not in df.columns:
    raise KeyError(f"Neither 'population' nor 'pop' was found in {data_source}.")
if len(df) < 1000:
    raise ValueError(f"The selected CSV has {len(df)} rows; at least 1,000 are required.")

print(f"Source: {data_source}")
print(f"Rows: {len(df):,}; analyzing column: {column_name!r}")
display(df.head())

In [ ]:
def leading_digit_frequency(dataframe, column_name):
    """Count first significant digits 1-9 in a numeric DataFrame column.

    Values are converted to numbers; missing, non-numeric, infinite, and zero
    values are excluded. Absolute values are used, so negative amounts are
    handled by their magnitude.
    """
    values = pd.to_numeric(dataframe[column_name], errors="coerce").to_numpy(dtype=float)
    values = values[np.isfinite(values) & (values != 0)]

    frequencies = {digit: 0 for digit in range(1, 10)}
    for value in values:
        first_digit = int(
            np.format_float_scientific(abs(value), precision=14, unique=False, trim="-")[0]
        )
        frequencies[first_digit] += 1
    return frequencies

In [ ]:
def benford_distribution():
    """Return Benford's expected leading-digit percentages for digits 1-9."""
    digits = np.arange(1, 10)
    percentages = np.log10(1 + 1 / digits) * 100
    return {int(digit): float(percent) for digit, percent in zip(digits, percentages)}

In [ ]:
def frequency_to_percent(frequencies):
    """Convert digit counts to percentages, retaining all keys provided."""
    total = sum(frequencies.values())
    if total == 0:
        raise ValueError("No non-zero numeric values are available for analysis.")
    return {digit: count / total * 100 for digit, count in frequencies.items()}

In [ ]:
def display_side_by_side(counts, actual, expected):
    """Display observed counts and percentages beside Benford percentages."""
    comparison = pd.DataFrame({
        "actual_frequency": [counts[digit] for digit in range(1, 10)],
        "actual_percent": [actual[digit] for digit in range(1, 10)],
        "benford_percent": [expected[digit] for digit in range(1, 10)],
    }, index=pd.Index(range(1, 10), name="first_digit"))
    comparison["difference_percentage_points"] = (
        comparison["actual_percent"] - comparison["benford_percent"]
    )
    display(comparison.round(4))
    return comparison

In [ ]:
def mad_statistic(actual, expected):
    """Compute mean absolute deviation across the nine digit percentages.

    The result is in percentage points because both inputs are percentages.
    """
    differences = [abs(actual[digit] - expected[digit]) for digit in range(1, 10)]
    return float(np.mean(differences))

In [ ]:
def ks_statistic(actual, expected):
    """Compute the maximum cumulative percentage-point gap across digits 1-9.

    This is the two-sample cumulative-distribution distance, calculated
    directly without a statistics library; it does not return a p-value.
    """
    actual_cumulative = np.cumsum([actual[digit] for digit in range(1, 10)])
    expected_cumulative = np.cumsum([expected[digit] for digit in range(1, 10)])
    return float(np.max(np.abs(actual_cumulative - expected_cumulative)))

In [ ]:
digit_counts = leading_digit_frequency(df, column_name)
actual_percent = frequency_to_percent(digit_counts)
expected_percent = benford_distribution()
valid_observations = sum(digit_counts.values())

comparison = display_side_by_side(digit_counts, actual_percent, expected_percent)
mad = mad_statistic(actual_percent, expected_percent)
ks = ks_statistic(actual_percent, expected_percent)

print(f"Usable non-zero numeric observations: {valid_observations:,}")
print(f"MAD: {mad:.4f} percentage points")
print(f"KS:  {ks:.4f} percentage points")

largest_gaps = comparison.reindex(
    comparison["difference_percentage_points"].abs().sort_values(ascending=False).index
).head(3)
print("\nDigits with the largest observed gaps:")
display(largest_gaps.round(4))
print(
    "Investigate these digits first by checking the underlying records, "
    "comparing years and countries separately, and confirming how population "
    "values were collected or rounded. A deviation can reflect real data "
    "structure and is not, by itself, evidence of an error or fraud."
)

## Conclusions and Next Steps

Use the computed table and statistics above to describe how this population sample compares with Benford's expected distribution. The digits printed under **largest observed gaps** are the first digits to investigate because they contribute most to the observed discrepancy. Check the individual records for those digits, then repeat the analysis by year and by country to see whether a concentrated subset explains the difference. Also check for rounding, minimum/maximum population constraints, duplicated records, and how the dataset was assembled.

MAD is the average absolute difference across the nine digit percentages; KS is the largest absolute difference between the cumulative actual and expected percentages. Both are reported in percentage points here. These descriptive statistics do not provide a p-value or prove that a dataset is correct or incorrect. Population data can depart from Benford because of the way populations are measured, rounded, and grouped. Interpret the numbers in light of those data-generation limits, and compare against an appropriate benchmark before drawing stronger conclusions.

The analysis uses all available non-zero numeric values in the selected column. If using a different dataset, confirm that the column contains values spanning multiple orders of magnitude and is not artificially bounded; Benford analysis is not appropriate for every numeric dataset.